In [ ]:
import boto3
import pandas as pd
import numpy as np

In [ ]:
!pip install openpyxl

In [ ]:
!aws sts get-caller-identity

{
    "UserId": "AROA6GBMFLM2RZZLOZJAL:SageMaker",
    "Account": "975050201909",
    "Arn": "arn:aws:sts::975050201909:assumed-role/AmazonSageMaker-ExecutionRole-20260521T003756/SageMaker"
}


In [ ]:
!aws s3 ls

2026-05-22 03:46:10 sagemaker-us-east-2-975050201909


In [ ]:
!aws s3 cp storedata_total.xlsx s3://sagemaker-us-east-2-975050201909/data/storedata_total.xlsx

upload: ./storedata_total.xlsx to s3://sagemaker-us-east-2-975050201909/data/storedata_total.xlsx


In [ ]:
## Preprocess the dataset
def preprocess_data(file_path):  
  df = pd.read_excel(file_path)
  ## Convert to datetime columns
  df["firstorder"]=pd.to_datetime(df["firstorder"],errors='coerce')
  df["lastorder"] = pd.to_datetime(df["lastorder"],errors='coerce')
  ## Drop Rows with null values
  df = df.dropna()    
  ## Create Column which gives the days between the last order and the first order
  df["first_last_days_diff"] = (df['lastorder']-df['firstorder']).dt.days
  ## Create Column which gives the days between when the customer record was created and the first order
  df['created'] = pd.to_datetime(df['created'])
  df['created_first_days_diff']=(df['created']-df['firstorder']).dt.days
  ## Drop Columns
  df.drop(['custid','created','firstorder','lastorder'],axis=1,inplace=True)
  ## Apply one hot encoding on favday and city columns
  df = pd.get_dummies(df,prefix=['favday','city'],columns=['favday','city'])
  return df

In [ ]:
default_bucket = "sagemaker-us-east-2-975050201909"
storedata = preprocess_data(f"s3://{default_bucket}/data/storedata_total.xlsx")

/opt/conda/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [ ]:
## Set the required configurations
model_name = "churn_model"
env = "dev"
## S3 Bucket
#default_bucket = "customer-churn-sm-pipeline"
## Preprocess the dataset
#storedata = preprocess_data(f"s3://{default_bucket}/data/storedata_total.xlsx")

In [ ]:
def split_datasets(df):
    y=df.pop("retained")
    X_pre = df
    y_pre = y.to_numpy().reshape(len(y),1)
    feature_names = list(X_pre.columns)
    X= np.concatenate((y_pre,X_pre),axis=1)
    np.random.shuffle(X)
    train,validation,test=np.split(X,[int(.7*len(X)),int(.85*len(X))])
    return feature_names,train,validation,test

# Split dataset
feature_names,train,validation,test = split_datasets(storedata)

# Save datasets in Amazon S3
pd.DataFrame(train).to_csv(f"s3://{default_bucket}/data/train/train.csv",header=False,index=False)
pd.DataFrame(validation).to_csv(f"s3://{default_bucket}/data/validation/validation.csv",header=False,index=False)
pd.DataFrame(test).to_csv(f"s3://{default_bucket}/data/test/test.csv",header=False,index=False)

In [ ]:
region = boto3.Session().region_name

In [ ]:
# Training and Validation Input for SageMaker Training job
from sagemaker.inputs import TrainingInput
import sagemaker
s3_input_train = TrainingInput(
    s3_data=f"s3://{default_bucket}/data/train/",content_type="csv")
s3_input_validation = TrainingInput(
    s3_data=f"s3://{default_bucket}/data/validation/",content_type="csv")

# Hyperparameter used
fixed_hyperparameters = {
    "eval_metric":"auc",
    "objective":"binary:logistic",
    "num_round":"100",
    "rate_drop":"0.3",
    "tweedie_variance_power":"1.4"
}

# Use the built-in SageMaker algorithm

sess = sagemaker.Session()
container = sagemaker.image_uris.retrieve("xgboost",region,"0.90-2")

from sagemaker import get_execution_role
from sagemaker.estimator import Estimator

role = get_execution_role()

estimator = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    hyperparameters=fixed_hyperparameters,
    instance_type="ml.m4.xlarge",
    output_path="s3://{}/output".format(default_bucket),
    sagemaker_session=sess
)
from sagemaker.tuner import (
    HyperparameterTuner,
    IntegerParameter,
    ContinuousParameter
)
hyperparameter_ranges = {
    "eta": ContinuousParameter(0, 1),
    "min_child_weight": ContinuousParameter(1, 10),
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
}
objective_metric_name = "validation:auc"
tuner = HyperparameterTuner(
    estimator=estimator, 
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,
    max_parallel_jobs=2
)

# Tune
tuner.fit({"train": s3_input_train, "validation": s3_input_validation})
tuner.wait()

## Explore the best model generated
tuning_job_result = boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name
)

best_training_job = tuner.best_training_job()
print(best_training_job)

job_count = tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" %job_count)
## 10 training jobs have completed

## Get the best training job

from pprint import pprint
if tuning_job_result.get("BestTrainingJob",None):
    print("Best Model found so far:")
    pprint(tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


.

.

.

...

.

.

!

!

sagemaker-xgboost-260526-1842-008-363bcf25
10 training jobs have completed
Best Model found so far:
{'CreationTime': datetime.datetime(2026, 5, 26, 18, 46, 55, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:auc',
                                                 'Value': 0.9797149896621704},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2026, 5, 26, 18, 47, 35, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-2:975050201909:training-job/sagemaker-xgboost-260526-1842-008-363bcf25',
 'TrainingJobName': 'sagemaker-xgboost-260526-1842-008-363bcf25',
 'TrainingJobStatus': 'Completed',
 'TrainingStartTime': datetime.datetime(2026, 5, 26, 18, 47, 1, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '1.594107174832447',
                          'eta': '0.08418707494060607',
                          'max_depth': '10',
                          'min_child_weight': '3.846002438912514'}}


In [ ]:
processing_instance_type = "ml.m5.large"
processing_instance_count = 1

In [ ]:
# processing step for feature engineering
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.processing import ProcessingInput, ProcessingOutput

processing_instance_type = "ml.m5.large"
processing_instance_count = 1

sklearn_processor = SKLearnProcessor(
        framework_version="0.23-1",
        instance_type=processing_instance_type,
        instance_count=processing_instance_count,
        sagemaker_session=sess,
        role=role,
    )

input_data = storedata

step_process = ProcessingStep(
        name="ChurnModelProcess",
        processor=sklearn_processor,
        inputs=[
          ProcessingInput(source=input_data, destination="/opt/ml/processing/input"),  
        ],
        outputs=[
            ProcessingOutput(output_name="train", source="/opt/ml/processing/train",\
                             destination=f"s3://{default_bucket}/output/train" ),
            ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation",\
                            destination=f"s3://{default_bucket}/output/validation"),
            ProcessingOutput(output_name="test", source="/opt/ml/processing/test",\
                            destination=f"s3://{default_bucket}/output/test")
        ],
        code=f"s3://{default_bucket}/input/code/preprocess.py",
    )

In [ ]:
# training step for generating model artifacts
model_path = f"s3://{default_bucket}/output"

training_instance_type = "ml.m4.xlarge"

image_uri = sagemaker.image_uris.retrieve(
        framework="xgboost",
        region=region,
        version="1.0-1",
        py_version="py3",
        instance_type=training_instance_type,
    )
fixed_hyperparameters = {
"eval_metric":"auc",
"objective":"binary:logistic",
"num_round":"100",
"rate_drop":"0.3",
"tweedie_variance_power":"1.4"
    }

xgb_train = Estimator(
        image_uri=image_uri,
        instance_type=training_instance_type,
        instance_count=1,
        hyperparameters=fixed_hyperparameters,
        output_path=model_path,
        base_job_name=f"churn-train",
        sagemaker_session=sess,
        role=role,
    )
hyperparameter_ranges = {
"eta": ContinuousParameter(0, 1),
"min_child_weight": ContinuousParameter(1, 10),
"alpha": ContinuousParameter(0, 2),
"max_depth": IntegerParameter(1, 10),
    }
objective_metric_name = "validation:auc"

In [ ]:
## Direct Integration for HPO

from sagemaker.workflow.steps import TuningStep

step_tuning = TuningStep(
name = "ChurnHyperParameterTuning",
tuner = HyperparameterTuner(xgb_train, objective_metric_name, hyperparameter_ranges, max_jobs=2, max_parallel_jobs=2),
inputs={
            "train": TrainingInput(
                s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                    "train"
                ].S3Output.S3Uri,
                content_type="text/csv",
            ),
            "validation": TrainingInput(
                s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                    "validation"
                ].S3Output.S3Uri,
                content_type="text/csv",
            ),
        },
    )

In [ ]:
sess = sagemaker.Session()
bucket = sess.default_bucket()

In [ ]:
# condition step for evaluating model quality and branching execution

from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.properties import PropertyFile

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)
script_eval = SKLearnProcessor(
    framework_version="0.23-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    sagemaker_session=sess
)
step_eval = ProcessingStep(
    name="ChurnModelEvaluation",
    processor=script_eval,
    inputs=[
        ProcessingInput(
            source=step_tuning.get_top_model_s3_uri(
                top_k=0,
                s3_bucket=bucket,
                prefix="top-model"
            ),
            destination="/opt/ml/processing/model"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        ),
    ],
    code="evaluate.py",
    property_files=[evaluation_report],
)
cond_lte = ConditionGreaterThan(
        left=JsonGet(
            step=step_eval,
            property_file=evaluation_report,
            json_path="classification_metrics.auc_score.value"
        ),
        right=0.75,
    )

In [ ]:
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_package_group_name = "ChurnModelPackageGroup"

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=f"{step_eval.arguments['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']}/evaluation.json",
        content_type="application/json",
    )
)

step_register = RegisterModel(
        name="RegisterChurnModel",
        estimator=xgb_train,
        model_data=step_tuning.get_top_model_s3_uri(top_k=0,s3_bucket=default_bucket,prefix="output"),
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.t2.medium", "ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name=model_package_group_name,
        model_metrics=model_metrics,
    )

Popping out 'ProcessingJobName' from the pipeline definition by default since it will be overridden at pipeline execution time. Please utilize the PipelineDefinitionConfig to persist this field in the pipeline definition if desired.


In [ ]:


data_config = sagemaker.clarify.DataConfig(
    s3_data_input_path=f's3://{sess.default_bucket}/output/train/train.csv',
    s3_output_path=f"s3://{default_bucket}.bias_report_output_path",
        label=0,
        headers= ['target','esent','eopenrate','eclickrate','avgorder','ordfreq','paperless','refill','doorstep','first_last_days_diff','created_first_days_diff','favday_Friday','favday_Monday','favday_Saturday','favday_Sunday','favday_Thursday','favday_Tuesday','favday_Wednesday','city_BLR','city_BOM','city_DEL','city_MAA'],
        dataset_type="text/csv",
    )
model_config = sagemaker.clarify.ModelConfig(
        model_name = "churn_model",
        instance_type = "ml.m5.large",
        instance_count=1,
        accept_type="text/csv",
    )
model_predicted_label_config = sagemaker.clarify.ModelPredictedLabelConfig(probability_threshold=0.5)
bias_config = sagemaker.clarify.BiasConfig(
        label_values_or_threshold=[1],
        facet_name="doorstep",
        facet_values_or_threshold=[0],
    )

In [ ]:
# step to perform batch transformation
from sagemaker.model import Model
from sagemaker.workflow.steps import CreateModelStep
from sagemaker.inputs import CreateModelInput

model = Model(
    image_uri=container,
    model_data=step_tuning.get_top_model_s3_uri(
        top_k=0,
        s3_bucket=default_bucket,
        prefix="top-model"
    ),
    role=role,
    sagemaker_session=sess,
)

step_create_model = CreateModelStep(
    name="CreateChurnModel",
    model=model,
    inputs=CreateModelInput(
        instance_type="ml.m5.large"
    )
)

from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep
from sagemaker.inputs import TransformInput

transformer = Transformer(
model_name=step_create_model.properties.ModelName,
instance_type="ml.m5.xlarge",
instance_count=1,
output_path=f"s3://{default_bucket}/ChurnTransform"
    )

batch_data = f"s3://{default_bucket}/batch/batch.csv"

step_transform = TransformStep(
name="ChurnTransform",
transformer=transformer,
inputs=TransformInput(data=batch_data,content_type="text/csv")
    )